# SAM2 vs YOLOv8 Experiments

Compares YOLOv8-only and YOLOv8+SAM2 pipelines on the same raster.

**Experiments:**
1. Detection count and confidence score distributions
2. Mask area distributions
3. Orientation distributions (key test — does SAM2 fix the pixel-grid artifact?)

In [ ]:
import sys
sys.path.insert(0, "/scratch/users/cayleigh/YOLOv8-BeyondEarth/src")
if "/home/users/cayleigh/BoulderNet/YOLOv8-BeyondEarth/src" in sys.path:
    sys.path.remove("/home/users/cayleigh/BoulderNet/YOLOv8-BeyondEarth/src")

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import Polygon
from shapely import segmentize
import geopandas as gpd
from pathlib import Path
from tqdm import tqdm

from sahi import AutoDetectionModel
from YOLOv8BeyondEarth.predict import get_sliced_predictionfast
from YOLOv8BeyondEarth.SAM2_predict import get_sliced_prediction_SAM2
from rastertools_BOULDERING import raster, metadata as raster_metadata
from shptools_BOULDERING.geometry import fitEllipse
from shptools_BOULDERING.geomorph import boulder_row

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

In [ ]:
home_p    = Path.home() / "Downloads" / "Test_ManualBoulderNetEnvironment"
work_dir  = home_p / "tmp" / "YOLOv8BeyondEarth"
in_raster = Path("/scratch/users/cayleigh/test_raster/M1221383405.tif")

yolo_output_dir = work_dir / "exp_yolo_256"
sam2_output_dir = work_dir / "exp_sam2_256"
yolo_output_dir.mkdir(parents=True, exist_ok=True)
sam2_output_dir.mkdir(parents=True, exist_ok=True)

# Load YOLO model
model_weights = work_dir / "yolov8_model" / "yolov8-m-boulder-detection-tmp.pt"
detection_model = AutoDetectionModel.from_pretrained(
    model_type='yolov8',
    model_path=model_weights.as_posix(),
    confidence_threshold=0.1,
    device="cuda:0",
    image_size=1024)

# Load SAM2 model
sam2_checkpoint = Path("/scratch/users/cayleigh/checkpoints/sam2.1_hiera_small.pt")
sam2_base_model = build_sam2("configs/sam2.1/sam2.1_hiera_s.yaml", sam2_checkpoint, device="cuda:0")
predictor = SAM2ImagePredictor(sam2_base_model)

In [ ]:
# Run YOLO inference (skip if output exists)
existing = list(yolo_output_dir.glob("*-downscaled-mask-nms.shp"))
if existing:
    print(f"YOLO output exists ({len(existing)} file(s)), skipping.")
else:
    print("Running YOLO inference...")
    t0 = time.time()
    get_sliced_predictionfast(
        in_raster,
        detection_model=detection_model,
        confidence_threshold=0.10,
        has_mask=True,
        output_dir=yolo_output_dir,
        slice_size=256,
        inference_size=1024,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2,
        min_area_threshold=6,
        downscale_pred=True,
        postprocess=True,
        postprocess_match_threshold=0.2,
        postprocess_class_agnostic=False,
        batch_size=16)
    print(f"Done in {(time.time()-t0)/60:.1f} min")

In [ ]:
# Run SAM2 inference (skip if output exists)
existing = list(sam2_output_dir.glob("*-downscaled-mask-nms.shp"))
if existing:
    print(f"SAM2 output exists ({len(existing)} file(s)), skipping.")
else:
    print("Running SAM2 inference...")
    t0 = time.time()
    get_sliced_prediction_SAM2(
        in_raster,
        predictor,
        detection_model=detection_model,
        confidence_threshold=0.10,
        output_dir=sam2_output_dir,
        slice_size=256,
        inference_size=1024,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2,
        min_area_threshold=6,
        downscale_pred=True,
        postprocess=True,
        postprocess_match_threshold=0.2,
        postprocess_class_agnostic=False,
        batch_size=16)
    print(f"Done in {(time.time()-t0)/60:.1f} min")

In [ ]:
# Load shapefiles
def load_shapefile(output_dir, glob="*-downscaled-mask-nms.shp"):
    shps = sorted(output_dir.glob(glob))
    assert shps, f"No shapefiles found in {output_dir}"
    gdfs = [gpd.read_file(p) for p in shps]
    gdf = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)
    gdf["poly_area"] = gdf.geometry.area
    return gdf

gdf_yolo = load_shapefile(yolo_output_dir)
gdf_sam2 = load_shapefile(sam2_output_dir)

res = raster_metadata.get_resolution(in_raster)[0]
areal_threshold = (res ** 2) * (4.74 ** 2)

gdf_yolo = gdf_yolo[gdf_yolo["poly_area"] >= areal_threshold].reset_index(drop=True)
gdf_sam2 = gdf_sam2[gdf_sam2["poly_area"] >= areal_threshold].reset_index(drop=True)

print(f"YOLO detections after area filter: {len(gdf_yolo)}")
print(f"SAM2 detections after area filter: {len(gdf_sam2)}")

## Experiment 1: Detection count and confidence score distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Detection counts
axes[0].bar(["YOLOv8", "YOLOv8+SAM2"], [len(gdf_yolo), len(gdf_sam2)],
            color=["steelblue", "tomato"])
axes[0].set_ylabel("Number of detections")
axes[0].set_title("Detection count comparison")
for i, v in enumerate([len(gdf_yolo), len(gdf_sam2)]):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Score distributions
axes[1].hist(gdf_yolo["score"], bins=30, alpha=0.6, color="steelblue", label="YOLOv8")
axes[1].hist(gdf_sam2["score"], bins=30, alpha=0.6, color="tomato", label="YOLOv8+SAM2")
axes[1].set_xlabel("Confidence score")
axes[1].set_ylabel("Count")
axes[1].set_title("Confidence score distribution")
axes[1].legend()

plt.tight_layout()
plt.savefig("exp1_detection_counts.png", dpi=150)
plt.show()

## Experiment 2: Mask area distributions

SAM2 should produce larger mask areas than YOLOv8 since it traces boulder boundaries more precisely.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (gdf, label, color) in zip(axes, [
    (gdf_yolo, "YOLOv8", "steelblue"),
    (gdf_sam2, "YOLOv8+SAM2", "tomato")
]):
    ax.hist(gdf["poly_area"], bins=50, color=color, edgecolor='k')
    ax.set_xlabel("Boulder area (m²)")
    ax.set_ylabel("Count")
    ax.set_title(f"{label} — mask area distribution\n"
                 f"median={gdf['poly_area'].median():.1f} m², "
                 f"mean={gdf['poly_area'].mean():.1f} m²")
    ax.set_xlim(0, gdf["poly_area"].quantile(0.99))

plt.suptitle("Mask area distributions — YOLOv8 vs SAM2")
plt.tight_layout()
plt.savefig("exp2_mask_areas.png", dpi=150)
plt.show()

print(f"YOLO  — median area: {gdf_yolo['poly_area'].median():.2f} m², mean: {gdf_yolo['poly_area'].mean():.2f} m²")
print(f"SAM2  — median area: {gdf_sam2['poly_area'].median():.2f} m², mean: {gdf_sam2['poly_area'].mean():.2f} m²")

## Experiment 3: Orientation distributions

YOLOv8 binary masks produce a staircase artifact that biases orientation measurements toward 90° and 180°.
SAM2's smoother masks should produce a flatter, less biased orientation distribution.

In [ ]:
def run_orientation_pipeline(gdf, res):
    records = []
    for _, row in tqdm(gdf.iterrows(), total=len(gdf)):
        poly = row.geometry
        row_seg = pd.Series({"geometry": segmentize(poly, res)})
        try:
            ellipse_poly, a_fit, b_fit, theta_rad = fitEllipse(row_seg)
            fit_failed = False
        except Exception:
            ellipse_poly, a_fit, b_fit, theta_rad = poly, 1.0, 1.0, 1.0
            fit_failed = True
        theta_deg = np.degrees(theta_rad)
        try:
            mrr_row = pd.Series({"geometry": ellipse_poly.minimum_rotated_rectangle})
            (_, _, long_axis, short_axis, _, _, _, angle180) = boulder_row(mrr_row)
            aspect_ratio = long_axis / short_axis if short_axis > 0 else None
        except Exception:
            angle180, aspect_ratio = None, None
        records.append({
            "theta_deg": theta_deg,
            "fit_failed": fit_failed,
            "angle180": angle180,
            "aspect_ratio": aspect_ratio,
            "poly_area": row["poly_area"],
        })
    return pd.DataFrame(records)

print("Running orientation pipeline on YOLO detections...")
df_yolo_orient = run_orientation_pipeline(gdf_yolo, res)
print("Running orientation pipeline on SAM2 detections...")
df_sam2_orient = run_orientation_pipeline(gdf_sam2, res)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for col, (df, label) in enumerate([(df_yolo_orient, "YOLOv8"), (df_sam2_orient, "YOLOv8+SAM2")]):
    df_elong = df[(df["aspect_ratio"] >= 1.2) & (df["aspect_ratio"] <= 2.0)]

    axes[0, col].hist(df_elong["theta_deg"].dropna(), bins=36, range=(0, 180),
                      color='tomato', edgecolor='k')
    axes[0, col].set_title(f"{label} — fitEllipse theta\n(elongated boulders, n={len(df_elong)})")
    axes[0, col].set_xlabel("theta (°)")
    axes[0, col].set_ylabel("count")

    axes[1, col].hist(df_elong["angle180"].dropna(), bins=36, range=(0, 180),
                      color='steelblue', edgecolor='k')
    axes[1, col].set_title(f"{label} — angle180\n(elongated boulders, n={len(df_elong)})")
    axes[1, col].set_xlabel("angle180 (°)")
    axes[1, col].set_ylabel("count")

plt.suptitle("Experiment 3: Orientation distributions — YOLOv8 vs YOLOv8+SAM2\n"
             "YOLOv8 should show spikes at 90°/180°; SAM2 should be flatter",
             fontsize=12)
plt.tight_layout()
plt.savefig("exp3_orientation_comparison.png", dpi=150)
plt.show()